# Lab 06: Returns
Reproduce Yahoo! Finance's adjusted close on a made-up price series, then use it to compute returns.

**Yahoo's method (two multipliers, applied to every price *before* the event date):**
- **Dividend multiplier** = 1 − dividend / (close on the day *before* the ex-date)
- **Split multiplier** = 1 / split ratio (a 4-for-1 split → 1/4)

## (a) Create the unadjusted closing price series

In [ ]:
# (a) Build the made-up price series (code given in the lab)
import pandas as pd
import numpy as np

dates = pd.date_range("2017-05-01", periods=6, freq="D")
x = pd.DataFrame({"close": [90.0, 92.0, 91.0, 93.0, 95.0, 23.0]}, index=dates)
x

## (b) Adjust for the $2 cash dividend (ex-date 2017-05-03)
The close the day before the ex-date (2017-05-02) is 92, so the multiplier is 1 − 2/92 ≈ 0.9783.
It applies to every close **before** 2017-05-03 (i.e., 05-01 and 05-02). Prices on and after the ex-date are unchanged.

In [ ]:
# (b) Dividend adjustment
dividend = 2.0
prev_close = x.loc["2017-05-02", "close"]        # close the day before the ex-date
div_multiplier = 1 - dividend / prev_close        # 1 - 2/92

x["adjusted"] = x["close"]                        # start with the raw close
x.loc[:"2017-05-02", "adjusted"] = x.loc[:"2017-05-02", "close"] * div_multiplier

print("Dividend multiplier:", div_multiplier)
x

## (c) Adjust for the 4-for-1 split (2017-05-06)
Multiply every **adjusted** close before 2017-05-06 by 1/4. The split-day price (23) stays as is.
Using the `adjusted` column (not `close`) keeps the dividend adjustment from (b).

In [ ]:
# (c) Split adjustment
split_multiplier = 1 / 4

x.loc[:"2017-05-05", "adjusted"] = x.loc[:"2017-05-05", "adjusted"] * split_multiplier
x

## (d) Total (simple) return
Total return = adjusted price today / adjusted price yesterday − 1. Using the adjusted close means dividends and splits are already accounted for.

In [ ]:
# (d) Total (simple) return from the adjusted close
x["total_return"] = x["adjusted"].pct_change()
x

## (e) Log return
Log return = ln(adjusted today / adjusted yesterday).

In [ ]:
# (e) Log return, then compare side by side with the total return
x["log_returns"] = np.log(x["adjusted"] / x["adjusted"].shift(1))

x[["total_return", "log_returns"]]

**Are they close?** Yes. Daily returns here are small (roughly −3% to +2%), and for small r, ln(1 + r) ≈ r.
The log return is always slightly below the simple return, and the gap widens as returns get larger
(e.g., the 05-06 move: simple ≈ −3.16% vs. log ≈ −3.21%).

## (f) Cumulative return
Cumulative return = product of (1 + daily total return) − 1. The first day has no return, so it is set to 0.

In [ ]:
# (f) Cumulative return, compounded from the daily total returns
x["cumulative_return"] = (1 + x["total_return"].fillna(0)).cumprod() - 1

# Sanity check: should equal last adjusted / first adjusted - 1
print("Check:", x["adjusted"].iloc[-1] / x["adjusted"].iloc[0] - 1)

x["cumulative_return"].plot(title="Cumulative Return", marker="o", grid=True)
x